# 04 — XGBoost · Overload Prediction

**Input:** `hive_metastore.gold.gold_features` (pre-scaled, with `split` column)  
**Tracking:** MLflow experiment `Transformer_Overload`

### Notebook structure

| Section | Description |
|---|---|
| 1 | Configuration & imports |
| 2 | Load data (train/test from `split` column) |
| 3 | Validation split for early stopping |
| 4 | Feature assembly pipeline |
| 5 | Hyperparameter grid search (with MLflow) |
| 6 | Best model evaluation on test set |
| 7 | Threshold sweep |
| 8 | Confusion matrix & curves |
| 9 | Feature importance (individual + grouped) |
| 10 | Summary |

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/03_gold_features/00_Evaluation

## 1 · Configuration & imports

In [0]:
import builtins
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, FeatureHasher
from xgboost.spark import SparkXGBClassifier
from pyspark.ml.functions import vector_to_array

import warnings
warnings.filterwarnings("ignore")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
SRC_TABLE    = "hive_metastore.gold.gold_features"
ID_COL       = "ID_prefix"
TS_COL       = "DATE"
LABEL_COL    = "label_4h"            # change to "label_24h" for 24 h horizon
MODEL_NAME   = "XGBoost"
EXPERIMENT   = "/Users/daniel.branco@cgi.com/Transformer_Overload_val"
SEED         = 42

# ── Feature groups (must match 02_gold_features) ─────────────────────────────
SIGNAL_COLS = ["current", "voltage"]

LOAD_RATIO_COLS = ["load_ratio_c", "load_ratio_v"]

ROLLING_COLS = [
    f"{s}_{stat}_{w}"
    for s in SIGNAL_COLS
    for stat in ["mean", "std", "max"]
    for w in ["1h", "1d", "7d"]
]

LAG_COLS = [
    f"{s}_lag_{l}"
    for s in SIGNAL_COLS
    for l in ["15m", "1h", "1d"]
]

WEATHER_RAW_COLS = [
    "temperatura_media_do_ar_horaria_c",
    "precipitacao_horaria_mm",
    "humidade_relativa_media_horaria_percent",
    "velocidade_do_vento_media_horaria_m_per_s",
]

WEATHER_DERIVED_COLS = ["temp_mean_1d", "temp_mean_7d", "precip_sum_1d"]

TEMPORAL_COLS = ["hour", "day_of_week", "month", "is_weekend"]

EVENT_COLS = ["events_15m_cnt"]

NUMERIC_FEATURE_COLS = (
    LOAD_RATIO_COLS
    + ROLLING_COLS
    + LAG_COLS
    + WEATHER_RAW_COLS
    + WEATHER_DERIVED_COLS
    + TEMPORAL_COLS
    + EVENT_COLS
)

CAT_COLS = [ID_COL, "CONCELHO"]

# Hash buckets for ID_prefix (1000+ unique values)
N_HASH_BUCKETS = 1024       # was 256; try 1024 if 2048 still OOMs
XGB_NUM_WORKERS = 24          # was defaultParallelism // 2 (likely 4–8); explicit cap for memory

# ── Sets used by classify_feature() in 00_Evaluation ─────────────────────────
WEATHER_RAW_SET     = set(WEATHER_RAW_COLS)
WEATHER_DERIVED_SET = set(WEATHER_DERIVED_COLS)
LOAD_RATIO_SET      = set(LOAD_RATIO_COLS)
TEMPORAL_SET        = set(TEMPORAL_COLS)
EVENT_SET           = set(EVENT_COLS)

# ── Validation split for early stopping ──────────────────────────────────────
# Last 20% of training rows (chronologically) used as validation
VAL_FRAC = 0.2

print(f"Numeric features : {len(NUMERIC_FEATURE_COLS)}")
print(f"Categorical cols : {CAT_COLS}")
print(f"Hash buckets     : {N_HASH_BUCKETS}")
print(f"Label            : {LABEL_COL}")

In [0]:
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "5000")

## 2 · Load data

In [0]:
df = spark.read.table(SRC_TABLE)
print(f"Loaded {df.count():,} rows  |  {len(df.columns)} columns")

keep_cols = NUMERIC_FEATURE_COLS + [LABEL_COL] + CAT_COLS + [TS_COL, "split"]

existing = set(df.columns)
missing = set(NUMERIC_FEATURE_COLS) - existing
if missing:
    print(f"⚠️  Missing columns (removed): {missing}")
    NUMERIC_FEATURE_COLS[:] = [c for c in NUMERIC_FEATURE_COLS if c in existing]

keep_cols = [c for c in keep_cols if c in existing]
df_clean = df.select(keep_cols).dropna()
print(f"After dropna: {df_clean.count():,} rows")

In [0]:
train_df = df_clean.filter(F.col("split") == "train")
test_df  = df_clean.filter(F.col("split") == "test")

print(f"Train : {train_df.count():,} rows")
print(f"Test  : {test_df.count():,} rows")

# Class imbalance ratio
n_train = train_df.count()
n_pos   = train_df.filter(F.col(LABEL_COL) == 1).count()
n_neg   = n_train - n_pos
scale_pos_weight = float(n_neg) / builtins.max(n_pos, 1)

print(f"\nTrain: {n_pos:,} pos ({100*n_pos/n_train:.2f}%)  |  {n_neg:,} neg")
print(f"scale_pos_weight = {scale_pos_weight:.4f}")

## 3 · Validation split for early stopping

XGBoost uses a `validation_indicator_col` (boolean column where `True` = validation row).  
We take the last 20% of training rows chronologically as validation.

In [0]:
# Find the chronological cutoff for validation (last 20% of training rows)
import datetime

total_train = train_df.count()

# Get min/max dates
date_range = train_df.agg(F.min(TS_COL).alias("min_dt"), F.max(TS_COL).alias("max_dt")).first()
min_dt = date_range["min_dt"]
max_dt = date_range["max_dt"]

# Cutoff = 80% of the way through the date range
total_seconds = (max_dt - min_dt).total_seconds()
val_cutoff = min_dt + datetime.timedelta(seconds=total_seconds * (1 - VAL_FRAC))
print(f"Validation cutoff: {val_cutoff}")

# Add is_val boolean column
train_w = train_df.withColumn(
    "is_val",
    F.col(TS_COL) >= F.lit(val_cutoff)
)

train_w = train_w.repartition(XGB_NUM_WORKERS).cache()
test_df  = test_df.repartition(XGB_NUM_WORKERS).cache()
print(f"train_w partitions: {train_w.rdd.getNumPartitions()}")
print(f"train_w rows      : {train_w.count():,}")

n_train_actual = train_w.filter(~F.col("is_val")).count()
n_val = train_w.filter(F.col("is_val")).count()
print(f"Train (actual) : {n_train_actual:,}")
print(f"Validation     : {n_val:,}")

print(f"\nCached ✅")

## 4 · Feature assembly pipeline

XGBoost is tree-based — scaling doesn't matter, but the data is pre-scaled anyway.  
Categoricals: FeatureHasher for `ID_prefix` (1000+ values), OHE for `CONCELHO` (18 values).

In [0]:
def make_xgb_pipeline(
    learning_rate=0.06, max_depth=6, subsample=0.8, colsample_bytree=0.6,
    num_round=1200, reg_lambda=2.0, min_child_weight=5, gamma=1.0,
    early_stopping=True,
):
    hasher = FeatureHasher(
        inputCols=CAT_COLS, outputCol="cat_hashed",
        numFeatures=N_HASH_BUCKETS, categoricalCols=CAT_COLS,
    )
    feat_assembler = VectorAssembler(
        inputCols=["cat_hashed"] + NUMERIC_FEATURE_COLS,
        outputCol="features", handleInvalid="skip",
    )
    xgb_kwargs = dict(
        features_col="features", label_col=LABEL_COL,
        prediction_col="prediction", probability_col="probability",
        raw_prediction_col="rawPrediction",
        learning_rate=learning_rate, max_depth=max_depth,
        subsample=subsample, colsample_bytree=colsample_bytree,
        num_round=num_round, reg_lambda=reg_lambda,
        min_child_weight=min_child_weight, gamma=gamma,
        scale_pos_weight=float(scale_pos_weight),
        eval_metric="aucpr", tree_method="hist",
        num_workers=builtins.max(1, spark.sparkContext.defaultParallelism // 2),
    )
    if early_stopping:
        xgb_kwargs["validation_indicator_col"] = "is_val"
        xgb_kwargs["early_stopping_rounds"]    = 100

    return Pipeline(stages=[hasher, feat_assembler, SparkXGBClassifier(**xgb_kwargs)])

## 5 · Hyperparameter grid search (with MLflow)

Each configuration gets its own nested MLflow run.  
Primary selection metric: **AUPRC** (evaluated on the held-out test set).

In [0]:
# (learning_rate, max_depth, colsample_bytree, min_child_weight, gamma, reg_lambda, num_round)
PARAM_GRID = [
    (0.10, 6, 0.6, 5,  1.0, 2.0, 800),
    (0.06, 6, 0.6, 10, 1.0, 2.0, 1200),
    (0.05, 6, 0.8, 8,  0.0, 3.0, 1500),
    (0.04, 8, 0.6, 12, 1.0, 3.0, 2000),
    (0.06, 8, 0.8, 5,  0.5, 1.0, 1000),
    (0.03, 6, 0.7, 10, 1.0, 2.0, 2500),
]

In [0]:
print(f"N_HASH_BUCKETS  = {N_HASH_BUCKETS}")
print(f"XGB_NUM_WORKERS = {XGB_NUM_WORKERS}")
print(f"train_w partitions = {train_w.rdd.getNumPartitions()}")
print(f"train_w rows       = {train_w.count():,}")
print(f"train_w cached?    = {train_w.is_cached}")

# Inspect the actual pipeline that will be used
test_pipe = make_xgb_pipeline(learning_rate=0.10, max_depth=6,
                              colsample_bytree=0.6, min_child_weight=5,
                              gamma=1.0, reg_lambda=2.0, num_round=800)
xgb_stage = test_pipe.getStages()[-1]
print(f"\nXGBoost stage params:")
print(f"  num_workers      = {xgb_stage.getOrDefault('num_workers')}")
print(f"  max_depth        = {xgb_stage.getOrDefault('max_depth')}")
print(f"  tree_method      = {xgb_stage.getOrDefault('tree_method')}")

In [0]:
sc = spark.sparkContext
print(f"defaultParallelism: {sc.defaultParallelism}")
print(f"Total executor cores: {sc._jsc.sc().getExecutorMemoryStatus().size()}")
# Cluster info
import json
try:
    ctx = json.loads(dbutils.notebook.entry_point.getDbutils().notebook().getContext().toJson())
    print(f"Cluster: {ctx.get('tags', {}).get('clusterName', 'unknown')}")
except Exception:
    pass

In [0]:
mlflow.set_experiment(EXPERIMENT)

best = {"auprc": -1, "run_id": None, "params": None, "best_iter": None}

val_df = train_w.filter(F.col("is_val"))  # same validation rows used for early stopping

with mlflow.start_run(run_name=f"{MODEL_NAME}_{LABEL_COL}") as parent_run:
    mlflow.log_param("model_type", MODEL_NAME)
    mlflow.log_param("label", LABEL_COL)
    mlflow.log_param("n_numeric_features", len(NUMERIC_FEATURE_COLS))
    mlflow.log_param("cat_cols", str(CAT_COLS))
    mlflow.log_param("scale_pos_weight", float(f"{scale_pos_weight:.4f}"))
    mlflow.log_param("n_hash_buckets", N_HASH_BUCKETS)
    mlflow.log_param("val_frac", VAL_FRAC)
    mlflow.log_param("selection_metric", "AUPRC_on_validation")

    for lr, md, csbt, mcw, gma, rl2, nr in PARAM_GRID:
        run_name = f"lr={lr}_md={md}_csbt={csbt}_mcw={mcw}"

        with mlflow.start_run(run_name=run_name, nested=True) as child_run:
            mlflow.log_param("learning_rate", lr)
            mlflow.log_param("max_depth", md)
            mlflow.log_param("colsample_bytree", csbt)
            mlflow.log_param("min_child_weight", mcw)
            mlflow.log_param("gamma", gma)
            mlflow.log_param("reg_lambda", rl2)
            mlflow.log_param("num_round_max", nr)

            pipe  = make_xgb_pipeline(
                learning_rate=lr, max_depth=md, colsample_bytree=csbt,
                min_child_weight=mcw, gamma=gma, reg_lambda=rl2, num_round=nr,
                early_stopping=True,
            )
            model = pipe.fit(train_w)

            # Best round from early stopping (fallback to nr if not available)
            try:
                booster   = model.stages[-1].get_booster()
                best_iter = getattr(booster, "best_iteration", None)
                effective_rounds = int(best_iter) + 1 if best_iter is not None else int(nr)
            except Exception:
                effective_rounds = int(nr)
            mlflow.log_metric("best_iteration", effective_rounds)

            # Evaluate on VAL
            val_preds = extract_prob_positive(model.transform(val_df))
            local     = val_preds.select(F.col(LABEL_COL).cast("int"), "prob_pos").toPandas()
            y_val, p_val = local[LABEL_COL].values, local["prob_pos"].values

            val_metrics = evaluate_binary(y_val, p_val, threshold=0.5,
                                          title=f"{MODEL_NAME} | {run_name} | VAL")
            log_evaluation_to_mlflow(val_metrics, y_val, p_val, threshold=0.5, prefix="val")

            if val_metrics["AUPRC"] > best["auprc"]:
                best = {"auprc": val_metrics["AUPRC"],
                        "run_id": child_run.info.run_id,
                        "params": (lr, md, csbt, mcw, gma, rl2, nr),
                        "best_iter": effective_rounds}

    mlflow.log_param("best_params",   str(best["params"]))
    mlflow.log_metric("best_val_AUPRC", best["auprc"])
    mlflow.log_metric("best_iter",      best["best_iter"])

print(f"\n🏆 Best on VAL: {best['params']} | best_iter={best['best_iter']} | val AUPRC={best['auprc']:.4f}")

## 6 · Best model — full evaluation on test set

In [0]:
# # Load best LR pipeline from MLflow instead of retraining
# import mlflow

# # Auto-discover the best run for this label
# exp = mlflow.get_experiment_by_name("/Users/daniel.branco@cgi.com/Transformer_Overload")
# runs = mlflow.search_runs(
#     experiment_ids=[exp.experiment_id],
#     filter_string=f"params.label = '{LABEL_COL}' and params.model_type = '{MODEL_NAME}'",
#     order_by=["metrics.best_AUPRC DESC"],
#     max_results=1,
# )
# RUN_ID = runs.iloc[0]["run_id"]
# print(f"Loading {MODEL_NAME} {LABEL_COL} from run {RUN_ID}")

# best_model = mlflow.spark.load_model(f"runs:/{RUN_ID}/best_xgb_model")
# best = {"model": best_model}  # match the variable name the rest of the notebook expects

In [0]:
while mlflow.active_run() is not None:
    mlflow.end_run()

In [0]:
lr_b, md_b, csbt_b, mcw_b, gma_b, rl2_b, _ = best["params"]
final_rounds = best["best_iter"]

# Drop the validation flag for the full-train refit
train_full = train_w.drop("is_val") if "is_val" in train_w.columns else train_w

with mlflow.start_run(run_id=parent_run.info.run_id):
    final_pipe  = make_xgb_pipeline(
        learning_rate=lr_b, max_depth=md_b, colsample_bytree=csbt_b,
        min_child_weight=mcw_b, gamma=gma_b, reg_lambda=rl2_b,
        num_round=final_rounds, early_stopping=False,
    )
    final_model = final_pipe.fit(train_full)

    test_preds = extract_prob_positive(final_model.transform(test_df))
    local      = test_preds.select(F.col(LABEL_COL).cast("int"), "prob_pos").toPandas()
    y_true = local[LABEL_COL].values
    y_prob = local["prob_pos"].values

    final_metrics = evaluate_binary(y_true, y_prob, threshold=0.5,
                                    title=f"FINAL TEST — {MODEL_NAME} ({LABEL_COL})")
    log_evaluation_to_mlflow(final_metrics, y_true, y_prob, threshold=0.5, prefix="test")
    with mlflow.start_run(run_name="final_refit", nested=True):
        mlflow.spark.log_model(final_model, artifact_path="model")
        
print(f"\nTest AUPRC: {final_metrics['AUPRC']:.4f}")

## 7 · Threshold sweep

In [0]:
sweep_results = threshold_sweep(y_true, y_prob)
display(spark.createDataFrame(sweep_results).orderBy(F.desc("f1")))

In [0]:
prec, rec, thrs = precision_recall_curve(y_true, y_prob)
f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-8)
best_idx = min(np.argmax(f1), len(thrs) - 1)

In [0]:
optimal_threshold = float(thrs[best_idx])
print(f"Optimal threshold (max F1): {optimal_threshold}")

optimal_metrics = evaluate_binary(y_true, y_prob, threshold=optimal_threshold, title=f"TEST @ threshold={optimal_threshold} — {MODEL_NAME}")

In [0]:
# best_thr_row = builtins.max(sweep_results, key=lambda r: r["f1"])
# optimal_threshold = best_thr_row["threshold"]
# print(f"Optimal threshold (max F1): {optimal_threshold}")

# optimal_metrics = evaluate_binary(y_true, y_prob, threshold=optimal_threshold, title=f"TEST @ threshold={optimal_threshold} — {MODEL_NAME}")

## 8 · Confusion matrix & curves

In [0]:
fig = plot_confusion_matrix(y_true, (y_prob >= 0.5).astype(int), title=f"{MODEL_NAME} — CM @ 0.5")
display(fig); plt.close(fig)

fig = plot_confusion_matrix(y_true, (y_prob >= optimal_threshold).astype(int), title=f"{MODEL_NAME} — CM @ {optimal_threshold}")
display(fig); plt.close(fig)

In [0]:
fig = plot_roc_curve(y_true, y_prob, title=f"{MODEL_NAME} — ROC")
display(fig); plt.close(fig)

In [0]:
fig = plot_pr_curve(y_true, y_prob, title=f"{MODEL_NAME} — Precision-Recall")
display(fig); plt.close(fig)

## 9 · Feature importance (individual + grouped)

For XGBoost, importance = **gain** (total reduction in loss contributed by each feature).  
Grouping and plotting functions come from `00_Evaluation`.

In [0]:
# ── XGBoost-specific: extract gain-based importance ──────────────────────────
from xgboost import XGBClassifier
best_model = final_model
# Get the native XGBoost booster from the Spark model
xgb_spark_model = None
for stage in best_model.stages:
    if hasattr(stage, "get_booster"):
        xgb_spark_model = stage
        break

assert xgb_spark_model is not None, "Could not find XGBoost stage in pipeline"

booster = xgb_spark_model.get_booster()
importance_dict = booster.get_score(importance_type="gain")

print(f"Features with non-zero gain: {len(importance_dict)}")

# ── Recover feature names from VectorAssembler metadata ──────────────────────
pred_one = best_model.transform(test_df.limit(1))
feat_meta = pred_one.schema["features"].metadata

feat_names = []
if "ml_attr" in feat_meta and "attrs" in feat_meta["ml_attr"]:
    for attr_type in feat_meta["ml_attr"]["attrs"]:
        for attr in feat_meta["ml_attr"]["attrs"][attr_type]:
            feat_names.append((attr["idx"], attr["name"]))
    feat_names.sort(key=lambda x: x[0])
    feat_names = [n for _, n in feat_names]
else:
    # Fallback: use booster feature names (f0, f1, ...)
    n_features = builtins.max(int(k.replace("f", "")) for k in importance_dict.keys()) + 1
    feat_names = [f"f{i}" for i in range(n_features)]

print(f"Feature names: {len(feat_names)}")

# ── Map booster importance (f0, f1, ...) to named features ───────────────────
importances = np.zeros(len(feat_names))
for key, gain in importance_dict.items():
    idx = int(key.replace("f", ""))
    if idx < len(importances):
        importances[idx] = float(gain)

print(f"Total gain: {float(np.sum(importances)):.2f}")

In [0]:
fig_top = plot_top_features(feat_names, importances, top_n=20, title=f"{MODEL_NAME} — Top 20 by Gain", xlabel="Gain")
display(fig_top)

# Table view
imp_rows = [
    (feat_names[i], float(importances[i]), classify_feature(feat_names[i]))
    for i in range(len(feat_names)) if importances[i] > 0
]
imp_df = spark.createDataFrame(imp_rows, ["feature", "gain", "group"])
display(imp_df.orderBy(F.desc("gain")).limit(20))

In [0]:
grp = grouped_importance(feat_names, importances)
fig_grp = plot_grouped_importance(grp, title=f"{MODEL_NAME} — Feature Group Importance", xlabel="Sum Gain")
display(fig_grp)

grp_rows = [(g, d["n_dims"], float(d["sum"]), float(d["mean"])) for g, d in grp.items()]
display(spark.createDataFrame(grp_rows, ["group", "n_dims", "sum_gain", "mean_gain"]).orderBy(F.desc("sum_gain")))

In [0]:
import pandas as pd

with mlflow.start_run(run_id=best["run_id"]):
    mlflow.log_figure(fig_top, "feature_importance_top20.png")
    mlflow.log_figure(fig_grp, "feature_importance_grouped.png")
    mlflow.log_table(
        pd.DataFrame([dict(group=g, **d) for g, d in grp.items()]),
        artifact_file="grouped_feature_importance.json",
    )

plt.close(fig_top)
plt.close(fig_grp)
print("✅ Feature importance logged to MLflow")